In [1]:
import re
from pathlib import Path
import joblib
import numpy as np
import pandas as pd

In [2]:
# ------------------------------------------------------------------------------
# Step 1: File Paths Setup
# ------------------------------------------------------------------------------
BASE_DIR = Path("/home/dell/Documents/Final_project_Car_Insurance_Claim_Prediction")
TEST_PATH = BASE_DIR / "data" / "raw" / "test.csv"
PROCESSED_DATA_PATH = BASE_DIR / "data" / "processed" / "split_data.pkl"
BEST_MODEL_PATH = BASE_DIR / "models" / "best_model.pkl"
SUBMISSION_PATH = BASE_DIR / "submission.csv"

In [4]:
print("--- Step 1: Loading Raw Test Data & Model Artifacts ---")
test_df = pd.read_csv(TEST_PATH)
data_artifacts = joblib.load(PROCESSED_DATA_PATH)

scaler = data_artifacts["scaler"]
train_features = data_artifacts["feature_names"]
best_model = joblib.load(BEST_MODEL_PATH)

# Save policy_id column for final output submission
policy_ids = (
    test_df["policy_id"]
    if "policy_id" in test_df.columns
    else test_df.index
)

print(f"Total Test Records Loaded: {len(test_df)}")
print(f"Loaded Champion Model:     {type(best_model).__name__}")

--- Step 1: Loading Raw Test Data & Model Artifacts ---
Total Test Records Loaded: 39063
Loaded Champion Model:     LGBMClassifier


In [5]:
# ------------------------------------------------------------------------------
# Step 2: Preprocess Test Features
# ------------------------------------------------------------------------------
print("\n--- Step 2: Preprocessing Test Features ---")


def extract_torque_value(text):
    match = re.search(r"([\d\.]+)Nm", str(text))
    return float(match.group(1)) if match else np.nan


def extract_power_value(text):
    match = re.search(r"([\d\.]+)bhp", str(text))
    return float(match.group(1)) if match else np.nan


--- Step 2: Preprocessing Test Features ---


In [6]:
# Drop policy_id column if present
if "policy_id" in test_df.columns:
    test_df = test_df.drop(columns=["policy_id"])

# Extract numeric torque and power
if "max_torque" in test_df.columns:
    test_df["torque_nm"] = test_df["max_torque"].apply(extract_torque_value)
    test_df = test_df.drop(columns=["max_torque"])

if "max_power" in test_df.columns:
    test_df["power_bhp"] = test_df["max_power"].apply(extract_power_value)
    test_df = test_df.drop(columns=["max_power"])

In [7]:
# Encode binary Yes/No flags to 1/0
binary_columns = [col for col in test_df.columns if col.startswith("is_")]
for col in binary_columns:
    if test_df[col].dtype == "object":
        test_df[col] = test_df[col].map({"Yes": 1, "No": 0})

# Log-transform population density
if "population_density" in test_df.columns:
    test_df["population_density"] = np.log1p(test_df["population_density"])

In [8]:
# Categorical One-Hot Encoding
categorical_cols = test_df.select_dtypes(include=["object"]).columns.tolist()
test_encoded = pd.get_dummies(test_df, columns=categorical_cols, drop_first=True)

# Align columns with training feature set (fill missing dummy columns with 0)
test_encoded = test_encoded.reindex(columns=train_features, fill_value=0)
test_encoded = test_encoded.fillna(test_encoded.median())

# Scale numerical features using fitted scaler
test_scaled = pd.DataFrame(
    scaler.transform(test_encoded), columns=train_features
)

In [9]:
# ------------------------------------------------------------------------------
# Step 3: Run Inference & Generate Submission File
# ------------------------------------------------------------------------------
print("\n--- Step 3: Generating Model Predictions ---")

# Predict binary labels (0 or 1) and continuous probabilities
predictions = best_model.predict(test_scaled)

if hasattr(best_model, "predict_proba"):
    probabilities = best_model.predict_proba(test_scaled)[:, 1]
else:
    probabilities = predictions


--- Step 3: Generating Model Predictions ---


In [10]:
# Format final dataframe
submission_df = pd.DataFrame({
    "policy_id": policy_ids,
    "is_claim": predictions,
    "claim_probability": np.round(probabilities, 4),
})

# Save output to CSV (only policy_id and predicted target as required by competitions)
submission_df[["policy_id", "is_claim"]].to_csv(SUBMISSION_PATH, index=False)

print("\n==========================================================================")
print("                       INFERENCE COMPLETED                                ")
print("==========================================================================")
print(f"📁 Output Submission File : {SUBMISSION_PATH}")
print(f"📊 Total Records Processed : {len(submission_df)}")
print(f"🔴 Predicted Claims (1)    : {sum(predictions == 1)} ({sum(predictions == 1)/len(predictions)*100:.2f}%)")
print(f"🔵 Predicted No-Claims (0) : {sum(predictions == 0)} ({sum(predictions == 0)/len(predictions)*100:.2f}%)")
print("\nFirst 5 rows of submission output:")
print(submission_df.head())
print("==========================================================================")


                       INFERENCE COMPLETED                                
📁 Output Submission File : /home/dell/Documents/Final_project_Car_Insurance_Claim_Prediction/submission.csv
📊 Total Records Processed : 39063
🔴 Predicted Claims (1)    : 17507 (44.82%)
🔵 Predicted No-Claims (0) : 21556 (55.18%)

First 5 rows of submission output:
  policy_id  is_claim  claim_probability
0   ID58593         1             0.7247
1   ID58594         1             0.5428
2   ID58595         0             0.3996
3   ID58596         0             0.4067
4   ID58597         0             0.3969
